# Лекция 6. Создание баз данных для хранения и анализа юридической информации

### Цель лекции

Изучить процесс проектирования и реализации баз данных для эффективного хранения и анализа юридических документов. Рассмотреть подходы к структурированию данных, выбор технологий баз данных, а также методы обеспечения целостности и безопасности данных.

---

### Введение

Информационно-поисковые системы в юридической сфере имеют дело с массивами данных, объём и сложность которых существенно превосходят возможности традиционного файлового хранения. Судебные акты, нормативно-правовые документы, материалы дел, экспертные заключения, апелляционные жалобы — все эти категории текстовых данных обладают как структурированными атрибутами (номер дела, дата вынесения решения, наименование суда, состав участников), так и обширными неструктурированными компонентами (полный текст решения, мотивировочная часть, особые мнения судей). Масштаб данных в этой области весьма значителен: только в системе ГАС «Правосудие» Российской Федерации ежегодно публикуется несколько миллионов судебных актов, а совокупный объём накопленного корпуса исчисляется десятками терабайт.

В таких условиях база данных выступает центральным инфраструктурным элементом любой юридической информационно-аналитической системы. Корректно спроектированная и реализованная база данных обеспечивает решение нескольких критически важных задач. Во-первых, она гарантирует структурированное и нормализованное хранение гетерогенных данных, объединяя метаданные, полные тексты документов, извлечённые именованные сущности и результаты аналитической обработки в единой логической модели. Во-вторых, база данных обеспечивает быстрый доступ к информации посредством индексирования и оптимизации запросов, что является необходимым условием для построения интерактивных поисковых сервисов с временем ответа, приемлемым для конечного пользователя. В-третьих, система управления базами данных реализует механизмы контроля целостности, конкурентного доступа, резервного копирования и разграничения прав, без которых эксплуатация системы в промышленном масштабе невозможна.

Выбор архитектуры хранения данных в юридических системах определяется рядом факторов, специфичных для данной предметной области. Юридические тексты отличаются значительной длиной (от нескольких до нескольких сотен страниц), сложной внутренней структурой (резолютивная, мотивировочная и описательная части), обилием ссылок на нормативные акты и прецедентные решения, а также необходимостью обеспечения юридической силы хранимых документов. Эти особенности предъявляют специфические требования к модели данных, механизмам индексирования и стратегиям масштабирования, которые будут последовательно рассмотрены в настоящей лекции.

Структура лекции охватывает полный жизненный цикл базы данных юридической информационной системы: от определения требований и проектирования концептуальной схемы через реализацию физической модели и настройку поисковых механизмов к обеспечению безопасности, масштабированию и эксплуатационному сопровождению. Каждый раздел сопровождается демонстрационным кодом на языках SQL и Python, иллюстрирующим изложенные концепции на конкретных примерах из юридической практики.

---


## 1. Основные понятия и определения

Прежде чем перейти к рассмотрению конкретных технологий и архитектурных решений, необходимо установить единый терминологический аппарат, на который будет опираться дальнейшее изложение. Строгость определений особенно важна в контексте проектирования баз данных, поскольку терминология в данной области исторически развивалась параллельно в нескольких научных школах и индустриальных сообществах, что нередко приводит к неоднозначности трактовок.

### 1.1. База данных

База данных (БД) представляет собой организованную совокупность взаимосвязанных данных, отражающих состояние объектов и их отношений в некоторой предметной области. Данное определение, восходящее к работам Э. Кодда (1970), подчёркивает два фундаментальных свойства базы данных: организованность (наличие определённой структуры, обеспечивающей эффективный доступ к данным) и взаимосвязанность (отражение отношений между объектами реального мира). В формальном смысле база данных может быть представлена как кортеж $\mathcal{B} = \langle S, D, C \rangle$, где $S$ — схема базы данных (совокупность определений структур), $D$ — множество данных, соответствующих схеме, а $C$ — множество ограничений целостности, которым данные должны удовлетворять.

В контексте юридических информационных систем база данных хранит как структурированные метаданные документов (идентификаторы, даты, типы, ссылки на участников), так и неструктурированные текстовые данные (полные тексты судебных актов, мотивировочные части решений, экспертные заключения). Такая дуальная природа данных предъявляет специфические требования к выбору модели данных и механизмов индексирования, поскольку эффективные подходы к обработке структурированных и неструктурированных данных существенно различаются.

Помимо хранения исходных документов, база данных юридической системы может содержать производные данные: результаты извлечения именованных сущностей (NER), тематические метки, присвоенные классификаторами, векторные представления документов (эмбеддинги), индексы полнотекстового поиска и агрегированные аналитические показатели. Наличие производных данных превращает базу данных из пассивного хранилища в активный аналитический инструмент, обеспечивающий не только поиск, но и многомерный анализ юридической информации.

### 1.2. Система управления базами данных (СУБД)

Система управления базами данных (СУБД) — программное обеспечение, обеспечивающее создание, поддержку и использование баз данных. СУБД выступает посредником между приложениями и физическим хранилищем данных, абстрагируя детали размещения данных на носителе и предоставляя высокоуровневый интерфейс для определения структуры данных, их манипулирования и администрирования. Архитектура современной СУБД, как правило, следует трёхуровневой модели ANSI/SPARC, включающей внешний уровень (пользовательские представления), концептуальный уровень (логическая модель данных) и внутренний уровень (физическое размещение данных).

Ключевыми функциями СУБД являются обеспечение транзакционности (выполнение свойств ACID — атомарность, согласованность, изолированность, долговечность), управление конкурентным доступом (механизмы блокировок и многоверсионного контроля), поддержка языка определения данных (DDL) и языка манипулирования данными (DML), а также предоставление средств администрирования и мониторинга. В контексте юридических систем особую значимость приобретает свойство долговечности (durability): утрата судебных актов или нарушение их целостности может иметь серьёзные правовые последствия, что обуславливает повышенные требования к механизмам журналирования и резервного копирования.

Формально свойства ACID можно выразить следующим образом. Пусть $T = \{t_1, t_2, \ldots, t_n\}$ — множество транзакций, выполняемых конкурентно. Атомарность гарантирует, что каждая транзакция $t_i$ выполняется либо целиком, либо не выполняется вовсе. Согласованность требует, чтобы после завершения каждой транзакции база данных переходила из одного согласованного состояния в другое, удовлетворяющее всем ограничениям $C$. Изолированность обеспечивает, что результат параллельного выполнения транзакций эквивалентен некоторому последовательному выполнению тех же транзакций. Долговечность гарантирует, что результаты успешно завершённой транзакции сохраняются даже в случае сбоя системы.

### 1.3. Реляционные и нереляционные базы данных

Реляционная модель данных, предложенная Э. Коддом в 1970 году, основана на математическом понятии отношения (relation). В реляционной базе данных информация организована в виде набора именованных отношений (таблиц), каждое из которых представляет собой множество кортежей (строк), обладающих одинаковой структурой. Формально отношение $R$ определяется на множестве доменов $D_1, D_2, \ldots, D_n$ как подмножество декартова произведения $R \subseteq D_1 \times D_2 \times \ldots \times D_n$. Каждый домен $D_i$ задаёт допустимое множество значений для соответствующего атрибута.

Реляционные базы данных обеспечивают строгую типизацию данных, поддержку ограничений целостности (первичные и внешние ключи, ограничения CHECK, уникальность), мощный декларативный язык запросов SQL, а также развитые механизмы оптимизации выполнения запросов. Эти свойства делают реляционные СУБД предпочтительным выбором для хранения структурированных метаданных юридических документов, где схема данных относительно стабильна и предсказуема.

Нереляционные базы данных (NoSQL) объединяют несколько принципиально различных моделей данных: документоориентированные (MongoDB, CouchDB), ключ-значение (Redis, DynamoDB), колоночные (Cassandra, HBase) и графовые (Neo4j, ArangoDB). Общей чертой NoSQL-систем является отказ от жёсткой схемы и табличной структуры в пользу более гибких моделей, допускающих хранение полуструктурированных и неструктурированных данных. В юридическом контексте документоориентированные базы данных представляют особый интерес, поскольку позволяют хранить судебные акты в их естественной иерархической структуре (разделы, параграфы, ссылки) без необходимости декомпозиции на плоские таблицы.

Выбор между реляционной и нереляционной моделью в юридических системах, как правило, не является бинарным. На практике наиболее эффективным оказывается гибридный подход: реляционная СУБД обеспечивает хранение структурированных метаданных и транзакционную обработку, а специализированные хранилища (документоориентированные, полнотекстовые или векторные) — работу с неструктурированным контентом и семантическим поиском.

### 1.4. Юридическая информация в контексте баз данных

Юридическая информация, подлежащая хранению в базе данных, характеризуется выраженной дуальной структурой. Структурированные данные включают метаданные документов — номер дела, дату вынесения решения, наименование суда, тип документа (решение, определение, постановление), сведения об участниках процесса, ссылки на применённые нормативные акты. Эти данные имеют фиксированную схему и допускают эффективное табличное хранение с поддержкой индексирования по произвольным комбинациям атрибутов.

Неструктурированные данные представляют собой полные тексты документов, объём которых варьируется от нескольких абзацев (определения о принятии иска) до сотен страниц (решения по сложным корпоративным спорам). Обработка таких текстов требует применения методов полнотекстового поиска, морфологического анализа и, в современных системах, векторного семантического поиска.

Промежуточное положение занимают полуструктурированные данные — результаты NLP-обработки документов: извлечённые именованные сущности с указанием типа и позиции в тексте, тематические метки, ключевые слова, рассчитанные оценки релевантности. Эти данные порождаются аналитическими конвейерами, рассмотренными в предшествующих лекциях курса, и подлежат хранению совместно с исходными документами для обеспечения целостности аналитического процесса.

---


## 2. Проектирование структуры базы данных

Проектирование базы данных представляет собой многоэтапный процесс, начинающийся с анализа предметной области и завершающийся физической реализацией на выбранной СУБД. Качество проектирования непосредственно определяет эффективность последующей эксплуатации системы: ошибки на этапе концептуального моделирования приводят к структурным проблемам, исправление которых в промышленной среде сопряжено со значительными затратами.

### 2.1. Определение требований

Процесс проектирования начинается с формализации требований к системе хранения данных. В контексте юридической информационно-поисковой системы основные функциональные требования можно систематизировать следующим образом.

Система должна обеспечивать хранение полных текстов судебных актов с сохранением исходного форматирования и структуры документа. Метаданные документов, включающие номер дела, дату, наименование суда, тип документа и сведения об участниках, подлежат хранению в структурированной форме с поддержкой эффективного поиска по произвольным комбинациям атрибутов. Система должна поддерживать связи между документами, делами и извлечёнными сущностями, отражая сложную структуру юридических отношений. Наконец, должна быть обеспечена возможность быстрого поиска документов как по структурированным параметрам, так и по содержанию текста.

К нефункциональным требованиям относятся производительность (время ответа на типичный поисковый запрос не должно превышать нескольких секунд при объёме базы в миллионы документов), масштабируемость (архитектура должна допускать горизонтальное расширение при росте объёма данных), надёжность (обеспечение сохранности данных при аппаратных сбоях) и безопасность (разграничение доступа, шифрование конфиденциальных данных, аудит операций).

Формализация требований может быть выполнена с использованием методологии UML (Unified Modeling Language), в рамках которой функциональные требования представляются диаграммами вариантов использования (use case diagrams), а информационная модель — диаграммами классов. Для юридической предметной области типичными акторами являются аналитик (осуществляет поиск и анализ документов), администратор (управляет структурой и правами доступа), автоматизированный конвейер (выполняет пакетную обработку и аннотирование документов).

### 2.2. Выбор типа базы данных

Выбор типа базы данных определяется характером хранимых данных, паттернами доступа и требованиями к производительности. Реляционные СУБД являются оптимальным выбором для хранения структурированных метаданных, где схема данных стабильна, а запросы включают сложные соединения (JOIN) нескольких таблиц. Строгая типизация и ограничения целостности реляционной модели обеспечивают консистентность данных, что особенно важно при работе с юридической информацией, где ошибки в метаданных могут привести к некорректным результатам поиска.

Нереляционные СУБД, напротив, предпочтительны для хранения неструктурированного текстового контента и полуструктурированных результатов аналитической обработки. Документоориентированные базы данных позволяют хранить документы различной структуры в единой коллекции без необходимости предварительного определения жёсткой схемы. Это преимущество особенно ценно при работе с юридическими документами, структура которых может существенно варьироваться в зависимости от типа акта, юрисдикции и исторического периода.

Наиболее обоснованным решением для юридической информационно-поисковой системы является гибридная архитектура, сочетающая реляционное хранилище для метаданных с специализированными компонентами для текстового поиска. PostgreSQL, поддерживающая как реляционную модель, так и расширения для полнотекстового поиска и хранения JSON-данных, представляет собой универсальное решение, позволяющее реализовать гибридный подход в рамках единой СУБД. При необходимости семантического поиска архитектура дополняется векторным хранилищем (FAISS, Qdrant, pgvector).

### 2.3. Проектирование схемы базы данных

#### 2.3.1. Определение сущностей и отношений

Концептуальная модель юридической базы данных строится на основе выделения ключевых сущностей предметной области и установления отношений между ними. Методология проектирования «сущность-связь» (Entity-Relationship, ER), предложенная П. Ченом в 1976 году, обеспечивает формальный аппарат для графического и текстового описания информационной модели.

Центральной сущностью модели является документ (Document) — юридический акт, подлежащий хранению и анализу. Атрибуты сущности Document включают уникальный идентификатор ($id$), заголовок ($title$), дату составления ($date$), полный текст ($text$) и тип документа ($type$). Сущность «судебное дело» (Case) описывает совокупность процессуальных событий, объединённых общим номером дела, и характеризуется атрибутами: номер дела ($number$), наименование суда ($court$), сведения об участниках ($participants$). Сущность «извлечённая сущность» (Entity) представляет именованную сущность, идентифицированную в тексте документа NER-моделью, и включает имя ($name$), тип ($type$) и позицию в тексте.

Между сущностями устанавливаются следующие отношения. Связь Document-Case имеет кардинальность «многие к одному»: несколько документов (решение, определение, апелляционное постановление) могут относиться к одному делу. Связь Document-Entity имеет кардинальность «один ко многим»: один документ содержит множество извлечённых сущностей. Связь Case-Entity является производной и реализуется через Document.

#### 2.3.2. Нормализация данных

Нормализация — процесс приведения структуры базы данных к нормальным формам с целью устранения избыточности данных и аномалий обновления. Теория нормализации, разработанная Э. Коддом и впоследствии расширенная Р. Бойсом, определяет иерархию нормальных форм, каждая из которых накладывает всё более строгие ограничения на допустимые функциональные зависимости между атрибутами.

Первая нормальная форма (1НФ) требует, чтобы все значения атрибутов были атомарными — каждая ячейка таблицы содержит ровно одно значение. В юридическом контексте это означает, что список участников дела не может храниться в виде единой строки в поле $participants$, а должен быть вынесен в отдельную таблицу с записью для каждого участника. Вторая нормальная форма (2НФ) исключает частичные функциональные зависимости: каждый неключевой атрибут функционально зависит от полного первичного ключа, а не от его части. Третья нормальная форма (3НФ) устраняет транзитивные зависимости: если атрибут $A$ определяет атрибут $B$, а $B$ определяет $C$, то $C$ должен быть вынесен в отдельную таблицу с ключом $B$.

На практике нормализация до 3НФ является стандартным требованием для транзакционных систем. Однако в аналитических системах, где преобладают операции чтения и агрегирования, допускается контролируемая денормализация с целью сокращения числа соединений и повышения производительности запросов. В юридической базе данных типичным примером денормализации является хранение текстового описания суда непосредственно в таблице дел, а не в отдельной справочной таблице.

---


In [ ]:
# ============================================================
# Демонстрация: концептуальная модель юридической БД
# Визуализация ER-диаграммы средствами matplotlib
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(1, 1, figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_title('ER-диаграмма юридической базы данных', fontsize=14, fontweight='bold')

# --- Сущности (прямоугольники) ---
entities_info = {
    'Case':     (1.0, 3.5, 3.0, 2.5, '#E3F2FD',
                 'Case (Дело)\n─────────────\nPK: id SERIAL\nnumber VARCHAR(50)\ncourt VARCHAR(255)\ndate_filed DATE\nstatus VARCHAR(30)'),
    'Document': (5.5, 3.5, 3.0, 2.5, '#FFF9C4',
                 'Document (Документ)\n─────────────────\nPK: id SERIAL\ntitle VARCHAR(255)\ndate DATE\ntext TEXT\ntype VARCHAR(50)\nFK: case_id → Case'),
    'Entity':   (10.0, 3.5, 3.0, 2.5, '#C8E6C9',
                 'Entity (Сущность)\n──────────────────\nPK: id SERIAL\nname VARCHAR(255)\ntype VARCHAR(50)\nstart_pos INTEGER\nFK: document_id → Document'),
}

for name, (x, y, w, h, color, label) in entities_info.items():
    rect = mpatches.FancyBboxPatch(
        (x, y), w, h, boxstyle="round,pad=0.1",
        edgecolor='#333333', facecolor=color, linewidth=1.5
    )
    ax.add_patch(rect)
    ax.text(x + w / 2, y + h / 2, label,
            ha='center', va='center', fontsize=8, family='monospace')

# --- Связи (стрелки) ---
# Case 1 ── N Document
ax.annotate('', xy=(5.5, 4.75), xytext=(4.0, 4.75),
            arrowprops=dict(arrowstyle='->', lw=2, color='#1565C0'))
ax.text(4.75, 5.2, '1 : N', ha='center', fontsize=10, color='#1565C0',
        fontweight='bold')

# Document 1 ── N Entity
ax.annotate('', xy=(10.0, 4.75), xytext=(8.5, 4.75),
            arrowprops=dict(arrowstyle='->', lw=2, color='#2E7D32'))
ax.text(9.25, 5.2, '1 : N', ha='center', fontsize=10, color='#2E7D32',
        fontweight='bold')

# --- Дополнительные сущности ---
# Participant
rect_p = mpatches.FancyBboxPatch(
    (1.0, 0.3), 3.0, 1.8, boxstyle="round,pad=0.1",
    edgecolor='#333333', facecolor='#F3E5F5', linewidth=1.5
)
ax.add_patch(rect_p)
ax.text(2.5, 1.2, 'Participant (Участник)\n───────────────────\nPK: id SERIAL\nname VARCHAR(255)\nrole VARCHAR(50)\nFK: case_id → Case',
        ha='center', va='center', fontsize=8, family='monospace')

ax.annotate('', xy=(2.5, 2.1), xytext=(2.5, 3.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='#7B1FA2'))
ax.text(3.2, 2.8, '1 : N', ha='center', fontsize=10, color='#7B1FA2',
        fontweight='bold')

# Annotation (тематическая метка)
rect_a = mpatches.FancyBboxPatch(
    (5.5, 0.3), 3.0, 1.8, boxstyle="round,pad=0.1",
    edgecolor='#333333', facecolor='#FBE9E7', linewidth=1.5
)
ax.add_patch(rect_a)
ax.text(7.0, 1.2, 'Annotation (Аннотация)\n───────────────────\nPK: id SERIAL\nlabel VARCHAR(100)\nconfidence FLOAT\nFK: document_id → Document',
        ha='center', va='center', fontsize=8, family='monospace')

ax.annotate('', xy=(7.0, 2.1), xytext=(7.0, 3.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='#BF360C'))
ax.text(7.7, 2.8, '1 : N', ha='center', fontsize=10, color='#BF360C',
        fontweight='bold')

plt.tight_layout()
plt.show()
print("ER-диаграмма юридической базы данных с 5 сущностями и связями")

## 3. Реализация базы данных

Переход от концептуальной модели к физической реализации представляет собой критический этап, на котором абстрактные сущности и отношения преобразуются в конкретные таблицы, столбцы, индексы и ограничения в рамках выбранной СУБД. Качество реализации непосредственно влияет на производительность, масштабируемость и удобство сопровождения системы.

### 3.1. Выбор СУБД

Выбор конкретной системы управления базами данных определяется совокупностью технических, экономических и организационных факторов. Среди реляционных СУБД PostgreSQL занимает особое положение в контексте юридических информационных систем благодаря ряду существенных преимуществ. PostgreSQL является свободно распространяемой СУБД с открытым исходным кодом, что исключает лицензионные затраты и снижает стоимость владения. Поддержка расширений (в частности, расширений для полнотекстового поиска, JSON-данных и векторных операций через pgvector) позволяет реализовать гибридный подход к хранению данных в рамках единой системы. Развитый механизм транзакций с поддержкой многоверсионного контроля конкурентного доступа (MVCC) обеспечивает высокую производительность при параллельном чтении и записи. Наконец, PostgreSQL поддерживает морфологический анализ для русского языка в модуле полнотекстового поиска, что является критически важным для обработки русскоязычных юридических текстов.

Среди нереляционных СУБД для юридических систем наибольший интерес представляет MongoDB — документоориентированная база данных, хранящая данные в формате BSON (Binary JSON). MongoDB позволяет хранить документы произвольной структуры в единой коллекции, что удобно для юридических актов различных типов и форматов. Однако отсутствие строгой схемы и транзакционной модели (в ранних версиях) ограничивает применимость MongoDB для задач, требующих гарантий целостности данных.

Для задач семантического поиска, рассмотренных в предыдущих лекциях курса, архитектура дополняется специализированным векторным хранилищем. Расширение pgvector для PostgreSQL позволяет хранить и индексировать высокоразмерные векторные представления документов непосредственно в реляционной СУБД, обеспечивая выполнение запросов приближённого поиска ближайших соседей (Approximate Nearest Neighbor, ANN). Сложность поиска $k$ ближайших соседей в наивной реализации составляет $O(n \cdot d)$, где $n$ — число документов, $d$ — размерность вектора, однако использование индексов HNSW (Hierarchical Navigable Small World) снижает её до $O(\log n)$ при приемлемой потере точности.

### 3.2. Создание таблиц и связей

Физическая реализация концептуальной модели в PostgreSQL осуществляется посредством языка определения данных (DDL). При создании таблиц необходимо тщательно подбирать типы данных для каждого столбца, устанавливать ограничения целостности и определять связи между таблицами посредством внешних ключей. Правильный выбор типов данных влияет не только на объём хранилища, но и на производительность запросов: например, использование типа $\texttt{VARCHAR(50)}$ вместо $\texttt{TEXT}$ для поля фиксированной длины позволяет СУБД более эффективно планировать операции сравнения и сортировки.

Ниже представлен полный DDL-скрипт для создания схемы юридической базы данных, включающий пять таблиц с ограничениями целостности, связями и комментариями.


In [ ]:
# ============================================================
# DDL-скрипт создания юридической базы данных
# Демонстрация полной схемы с ограничениями целостности
# ============================================================

# Формируем SQL-скрипт как строку для наглядности
ddl_script = """
-- ============================================================
-- Схема базы данных юридической информационно-поисковой системы
-- СУБД: PostgreSQL 15+
-- ============================================================

-- 1. Справочник судов
CREATE TABLE courts (
    id          SERIAL PRIMARY KEY,
    name        VARCHAR(255) NOT NULL,          -- полное наименование суда
    region      VARCHAR(100),                   -- регион
    level       VARCHAR(50)                     -- уровень: районный, апелляционный, кассационный
        CHECK (level IN ('районный', 'апелляционный', 'кассационный', 'верховный'))
);

-- 2. Судебные дела
CREATE TABLE cases (
    id          SERIAL PRIMARY KEY,
    number      VARCHAR(50) NOT NULL UNIQUE,    -- номер дела (уникален)
    court_id    INTEGER NOT NULL
        REFERENCES courts(id) ON DELETE RESTRICT,
    date_filed  DATE NOT NULL,                  -- дата подачи
    category    VARCHAR(100),                   -- категория: гражданское, уголовное и т.д.
    status      VARCHAR(30) DEFAULT 'в производстве'
        CHECK (status IN ('в производстве', 'завершено', 'на обжаловании'))
);

-- 3. Участники дела (нормализованная таблица, 1НФ)
CREATE TABLE participants (
    id          SERIAL PRIMARY KEY,
    case_id     INTEGER NOT NULL
        REFERENCES cases(id) ON DELETE CASCADE,
    name        VARCHAR(255) NOT NULL,
    role        VARCHAR(50) NOT NULL            -- истец, ответчик, третье лицо, судья
        CHECK (role IN ('истец', 'ответчик', 'третье лицо', 'судья', 'прокурор')),
    org_type    VARCHAR(20)                     -- физ. лицо, юр. лицо
);

-- 4. Документы
CREATE TABLE documents (
    id          SERIAL PRIMARY KEY,
    title       VARCHAR(500) NOT NULL,
    date        DATE NOT NULL,
    text        TEXT NOT NULL,                  -- полный текст документа
    doc_type    VARCHAR(50) NOT NULL
        CHECK (doc_type IN ('решение', 'определение', 'постановление',
                            'исковое заявление', 'апелляционная жалоба')),
    case_id     INTEGER NOT NULL
        REFERENCES cases(id) ON DELETE CASCADE,
    created_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- 5. Извлечённые именованные сущности (результаты NER)
CREATE TABLE entities (
    id          SERIAL PRIMARY KEY,
    document_id INTEGER NOT NULL
        REFERENCES documents(id) ON DELETE CASCADE,
    name        VARCHAR(255) NOT NULL,
    ner_type    VARCHAR(50) NOT NULL            -- PERSON, ORG, DATE, LAW_REF, MONEY
        CHECK (ner_type IN ('PERSON', 'ORG', 'DATE', 'LAW_REF', 'MONEY', 'CASE_NUM')),
    start_pos   INTEGER,                        -- начальная позиция в тексте
    end_pos     INTEGER,                        -- конечная позиция в тексте
    confidence  REAL DEFAULT 1.0                -- уверенность NER-модели
        CHECK (confidence BETWEEN 0.0 AND 1.0)
);

-- 6. Тематические аннотации документов
CREATE TABLE annotations (
    id          SERIAL PRIMARY KEY,
    document_id INTEGER NOT NULL
        REFERENCES documents(id) ON DELETE CASCADE,
    label       VARCHAR(100) NOT NULL,          -- тематическая метка
    confidence  REAL DEFAULT 1.0
        CHECK (confidence BETWEEN 0.0 AND 1.0),
    model_name  VARCHAR(100)                    -- модель, присвоившая метку
);

-- Индексы для ускорения типичных запросов
CREATE INDEX idx_documents_case    ON documents(case_id);
CREATE INDEX idx_documents_date    ON documents(date);
CREATE INDEX idx_documents_type    ON documents(doc_type);
CREATE INDEX idx_entities_doc      ON entities(document_id);
CREATE INDEX idx_entities_type     ON entities(ner_type);
CREATE INDEX idx_annotations_doc   ON annotations(document_id);
CREATE INDEX idx_cases_number      ON cases(number);
CREATE INDEX idx_participants_case ON participants(case_id);

-- Полнотекстовый индекс (GIN) для поиска по содержимому документов
CREATE INDEX idx_documents_fts
    ON documents
    USING gin(to_tsvector('russian', text));
"""

print("DDL-скрипт юридической базы данных")
print("=" * 60)
print(ddl_script)
print("\nСхема включает 6 таблиц, 8+ индексов, ограничения CHECK и внешние ключи.")
print("Полнотекстовый GIN-индекс обеспечивает морфологический поиск по русскоязычным текстам.")

### 3.3. Наполнение базы данных

Процесс наполнения базы данных юридическими документами может осуществляться несколькими способами в зависимости от источника данных и требуемого объёма загрузки. Для массового первоначального импорта применяется утилита $\texttt{COPY}$, обеспечивающая высокопроизводительную загрузку данных из файлов формата CSV непосредственно в таблицы базы. Производительность $\texttt{COPY}$ существенно превышает производительность поочерёдного выполнения операторов $\texttt{INSERT}$, поскольку данные загружаются блоками с минимальным числом транзакционных операций.

Для программного наполнения базы данных из приложений на языке Python используется библиотека SQLAlchemy, реализующая паттерн объектно-реляционного отображения (ORM). ORM обеспечивает абстракцию над SQL-операциями, позволяя работать с записями базы данных как с объектами языка программирования. Данный подход повышает читаемость и сопровождаемость кода, а также обеспечивает переносимость приложения между различными СУБД.

При наполнении базы необходимо обеспечить соблюдение порядка загрузки данных, определяемого зависимостями между таблицами (внешними ключами). Сначала загружаются справочные таблицы ($courts$), затем основные сущности ($cases$, $participants$), после чего — документы ($documents$) и, наконец, производные данные ($entities$, $annotations$). Нарушение порядка загрузки приведёт к ошибкам нарушения ограничений целостности.

### 3.4. Индексирование данных

Индекс — вспомогательная структура данных, обеспечивающая ускорение операций поиска и фильтрации за счёт дополнительного расхода дискового пространства и замедления операций вставки и обновления. В терминах вычислительной сложности индекс преобразует операцию последовательного сканирования таблицы со сложностью $O(n)$ в операцию поиска по дереву со сложностью $O(\log n)$ (для B-tree индексов) или поиска по инвертированному списку со сложностью $O(k)$ (для GIN-индексов полнотекстового поиска), где $n$ — число записей в таблице, $k$ — число документов, содержащих искомый термин.

PostgreSQL поддерживает несколько типов индексов, каждый из которых оптимален для определённого класса запросов. B-tree индексы обеспечивают эффективный поиск по точному совпадению и диапазонным условиям и являются типом индекса по умолчанию. GIN (Generalized Inverted Index) индексы предназначены для полнотекстового поиска и работы с массивами, обеспечивая быстрый поиск документов, содержащих заданные термины. GiST (Generalized Search Tree) индексы поддерживают поиск по пространственным данным и операции сходства. Расширение pgvector добавляет поддержку HNSW- и IVFFlat-индексов для приближённого поиска ближайших соседей в пространстве эмбеддингов.

Стратегия индексирования должна определяться анализом типичных запросов к системе. В юридической базе данных наиболее частыми являются запросы поиска документов по номеру дела, дате, типу документа, а также полнотекстовые запросы по содержанию. Для каждого из этих паттернов необходим соответствующий индекс. Вместе с тем избыточное индексирование приводит к замедлению операций записи и увеличению объёма хранилища, поэтому создание индексов должно быть обосновано анализом реальных паттернов нагрузки.

---


In [ ]:
# ============================================================
# Демонстрация: ORM-модели SQLAlchemy для юридической БД
# Работа с объектами Python вместо сырых SQL-запросов
# ============================================================
# Примечание: для выполнения кода необходимы:
#   pip install sqlalchemy
# Демонстрация использует SQLite в памяти для портативности

from sqlalchemy import (
    create_engine, Column, Integer, String, Text, Date,
    Float, ForeignKey, CheckConstraint, UniqueConstraint,
    func, Index
)
from sqlalchemy.orm import (
    declarative_base, relationship, sessionmaker
)
from datetime import date

# --- Инициализация движка (SQLite in-memory для демонстрации) ---
engine = create_engine('sqlite:///:memory:', echo=False)
Base = declarative_base()
Session = sessionmaker(bind=engine)

# ============================================================
# Определение ORM-моделей
# ============================================================

class Court(Base):
    """Справочник судов."""
    __tablename__ = 'courts'
    id     = Column(Integer, primary_key=True)
    name   = Column(String(255), nullable=False)
    region = Column(String(100))
    level  = Column(String(50))
    # Обратная связь: список дел данного суда
    cases  = relationship('Case', back_populates='court')


class Case(Base):
    """Судебное дело."""
    __tablename__ = 'cases'
    id          = Column(Integer, primary_key=True)
    number      = Column(String(50), nullable=False, unique=True)
    court_id    = Column(Integer, ForeignKey('courts.id'), nullable=False)
    date_filed  = Column(Date, nullable=False)
    category    = Column(String(100))
    status      = Column(String(30), default='в производстве')
    # Связи
    court        = relationship('Court', back_populates='cases')
    documents    = relationship('Document', back_populates='case',
                                cascade='all, delete-orphan')
    participants = relationship('Participant', back_populates='case',
                                cascade='all, delete-orphan')


class Participant(Base):
    """Участник судебного дела."""
    __tablename__ = 'participants'
    id      = Column(Integer, primary_key=True)
    case_id = Column(Integer, ForeignKey('cases.id'), nullable=False)
    name    = Column(String(255), nullable=False)
    role    = Column(String(50), nullable=False)
    # Связь
    case = relationship('Case', back_populates='participants')


class Document(Base):
    """Юридический документ (судебный акт)."""
    __tablename__ = 'documents'
    id       = Column(Integer, primary_key=True)
    title    = Column(String(500), nullable=False)
    date     = Column(Date, nullable=False)
    text     = Column(Text, nullable=False)
    doc_type = Column(String(50), nullable=False)
    case_id  = Column(Integer, ForeignKey('cases.id'), nullable=False)
    # Связи
    case        = relationship('Case', back_populates='documents')
    entities    = relationship('Entity', back_populates='document',
                               cascade='all, delete-orphan')
    annotations = relationship('Annotation', back_populates='document',
                               cascade='all, delete-orphan')


class Entity(Base):
    """Извлечённая именованная сущность (результат NER)."""
    __tablename__ = 'entities'
    id          = Column(Integer, primary_key=True)
    document_id = Column(Integer, ForeignKey('documents.id'), nullable=False)
    name        = Column(String(255), nullable=False)
    ner_type    = Column(String(50), nullable=False)
    start_pos   = Column(Integer)
    end_pos     = Column(Integer)
    confidence  = Column(Float, default=1.0)
    # Связь
    document = relationship('Document', back_populates='entities')


class Annotation(Base):
    """Тематическая аннотация документа."""
    __tablename__ = 'annotations'
    id          = Column(Integer, primary_key=True)
    document_id = Column(Integer, ForeignKey('documents.id'), nullable=False)
    label       = Column(String(100), nullable=False)
    confidence  = Column(Float, default=1.0)
    model_name  = Column(String(100))
    # Связь
    document = relationship('Document', back_populates='annotations')


# --- Создание всех таблиц ---
Base.metadata.create_all(engine)
print("Таблицы успешно созданы:")
for table_name in Base.metadata.tables:
    print(f"  • {table_name}")

# ============================================================
# Наполнение базы тестовыми данными
# ============================================================
session = Session()

# Суд
court1 = Court(name='Арбитражный суд города Москвы',
               region='Москва', level='районный')
session.add(court1)
session.flush()  # получаем id

# Дело
case1 = Case(number='А40-12345/2024', court_id=court1.id,
             date_filed=date(2024, 3, 15), category='банкротство',
             status='в производстве')
session.add(case1)
session.flush()

# Участники
p1 = Participant(case_id=case1.id, name='ООО «Альфа»', role='истец')
p2 = Participant(case_id=case1.id, name='ПАО «Бета-Банк»', role='ответчик')
session.add_all([p1, p2])

# Документ
doc1 = Document(
    title='Решение по делу А40-12345/2024 о признании сделки недействительной',
    date=date(2024, 6, 20),
    text=('Арбитражный суд города Москвы в составе судьи Ивановой А.Б., '
          'рассмотрев дело по иску ООО «Альфа» к ПАО «Бета-Банк» '
          'о признании недействительной сделки по перечислению денежных средств '
          'в размере 15 000 000 рублей, установил следующее. '
          'В ходе процедуры банкротства конкурсным управляющим было выявлено, '
          'что указанная сделка совершена с целью причинения вреда имущественным '
          'правам кредиторов в соответствии со статьёй 61.2 Федерального закона '
          '«О несостоятельности (банкротстве)».'),
    doc_type='решение',
    case_id=case1.id
)
session.add(doc1)
session.flush()

# Извлечённые сущности (имитация результатов NER)
ner_results = [
    Entity(document_id=doc1.id, name='ООО «Альфа»',
           ner_type='ORG', confidence=0.97),
    Entity(document_id=doc1.id, name='ПАО «Бета-Банк»',
           ner_type='ORG', confidence=0.95),
    Entity(document_id=doc1.id, name='Иванова А.Б.',
           ner_type='PERSON', confidence=0.92),
    Entity(document_id=doc1.id, name='15 000 000 рублей',
           ner_type='MONEY', confidence=0.98),
    Entity(document_id=doc1.id, name='ст. 61.2 ФЗ «О банкротстве»',
           ner_type='LAW_REF', confidence=0.94),
]
session.add_all(ner_results)

# Аннотация
ann1 = Annotation(document_id=doc1.id, label='банкротство',
                  confidence=0.91, model_name='LogReg-TfIdf-v1')
session.add(ann1)

session.commit()
print(f"\nДобавлено записей: 1 суд, 1 дело, 2 участника, "
      f"1 документ, {len(ner_results)} сущностей, 1 аннотация")

## 4. Поиск и анализ данных в базе

Эффективный поиск является ключевой функцией информационно-поисковой системы, определяющей её практическую ценность для конечного пользователя. В контексте юридической базы данных различаются три принципиально различных парадигмы поиска, каждая из которых ориентирована на определённый класс информационных потребностей.

### 4.1. Структурированные SQL-запросы

Структурированные запросы на языке SQL обеспечивают точный поиск по метаданным документов — номеру дела, дате, типу документа, наименованию суда, участникам процесса. Данный вид поиска предполагает, что пользователь точно знает, какие атрибуты характеризуют искомый документ, и может сформулировать запрос в терминах этих атрибутов.

Язык SQL предоставляет богатый набор конструкций для фильтрации, агрегирования и соединения данных. Оператор $\texttt{SELECT}$ извлекает столбцы, $\texttt{WHERE}$ задаёт условия фильтрации, $\texttt{JOIN}$ соединяет связанные таблицы, $\texttt{GROUP BY}$ и агрегатные функции ($\texttt{COUNT}$, $\texttt{SUM}$, $\texttt{AVG}$) обеспечивают вычисление сводных показателей. Подзапросы и общие табличные выражения (CTE) позволяют строить сложные многоуровневые запросы.

Оптимизатор запросов PostgreSQL автоматически выбирает наиболее эффективный план выполнения каждого запроса на основе статистики о распределении данных. Команда $\texttt{EXPLAIN ANALYZE}$ позволяет визуализировать план выполнения и оценить его стоимость в условных единицах. Понимание планов выполнения является необходимым навыком для диагностики и устранения проблем производительности.

### 4.2. Полнотекстовый поиск

Полнотекстовый поиск в PostgreSQL реализуется посредством типов данных $\texttt{tsvector}$ и $\texttt{tsquery}$ и оператора совпадения $\texttt{@@}$. Тип $\texttt{tsvector}$ представляет документ как упорядоченный список нормализованных лексем с указанием их позиций в исходном тексте. Тип $\texttt{tsquery}$ описывает поисковый запрос в виде комбинации лексем, связанных логическими операторами $\texttt{\&}$ (И), $\texttt{|}$ (ИЛИ) и $\texttt{!}$ (НЕ).

Нормализация лексем выполняется с учётом морфологии языка: для русского языка словоформы приводятся к словарной форме, что обеспечивает поиск документов, содержащих любые грамматические формы искомого слова. Например, запрос «банкротство» также находит документы, содержащие слова «банкротства», «банкротстве», «банкротству». Функция $\texttt{ts\_headline}$ формирует фрагмент текста (сниппет) с подсветкой найденных терминов, что облегчает визуальную оценку релевантности результатов.

Ранжирование результатов полнотекстового поиска осуществляется функцией $\texttt{ts\_rank}$, вычисляющей оценку релевантности на основе частоты и позиции терминов запроса в документе. Формула ранжирования основана на модели TF-IDF, адаптированной для работы с типами $\texttt{tsvector}$. Для более точного ранжирования доступна функция $\texttt{ts\_rank\_cd}$, учитывающая плотность (cover density) терминов запроса в документе, то есть близость найденных терминов друг к другу.

### 4.3. Интеграция поиска с приложением

Взаимодействие приложения с базой данных в промышленных системах реализуется посредством паттерна «объектно-реляционное отображение» (ORM), обеспечивающего прозрачное преобразование между объектами языка программирования и записями базы данных. Библиотека SQLAlchemy для Python является наиболее зрелой и функциональной реализацией данного паттерна в экосистеме Python.

SQLAlchemy предоставляет два уровня абстракции: Core (низкоуровневый, основанный на SQL-выражениях) и ORM (высокоуровневый, основанный на отображении классов на таблицы). ORM-уровень позволяет выполнять операции CRUD (Create, Read, Update, Delete) посредством стандартных операций с Python-объектами, а SQLAlchemy автоматически генерирует и выполняет соответствующие SQL-запросы. При этом разработчик сохраняет возможность использования сырых SQL-запросов в случаях, когда ORM-абстракция недостаточна.

Для построения REST API над базой данных используется фреймворк FastAPI, обеспечивающий автоматическую валидацию входных данных, генерацию документации (OpenAPI/Swagger) и асинхронную обработку запросов. Комбинация SQLAlchemy ORM и FastAPI представляет собой стандартный технологический стек для создания серверной части юридических информационных систем.

---


In [ ]:
# ============================================================
# Демонстрация: запросы к юридической базе данных
# Структурированный поиск, аналитические запросы, ORM-фильтры
# ============================================================

# --- 1. Поиск документов по номеру дела ---
print("=" * 60)
print("1. Поиск документов по номеру дела 'А40-12345/2024'")
print("=" * 60)

docs = (session.query(Document)
        .join(Case)
        .filter(Case.number == 'А40-12345/2024')
        .all())

for d in docs:
    print(f"  ID: {d.id} | {d.doc_type} | {d.date} | {d.title[:60]}...")

# --- 2. Получение участников дела ---
print("\n" + "=" * 60)
print("2. Участники дела")
print("=" * 60)

parts = (session.query(Participant)
         .join(Case)
         .filter(Case.number == 'А40-12345/2024')
         .all())

for p in parts:
    print(f"  {p.role}: {p.name}")

# --- 3. Извлечённые сущности из документа ---
print("\n" + "=" * 60)
print("3. Именованные сущности (NER) из документа")
print("=" * 60)

ents = (session.query(Entity)
        .filter(Entity.document_id == doc1.id)
        .order_by(Entity.ner_type)
        .all())

for e in ents:
    print(f"  [{e.ner_type:10s}] {e.name:<35s} (conf: {e.confidence:.2f})")

# --- 4. Аналитический запрос: количество сущностей по типам ---
print("\n" + "=" * 60)
print("4. Аналитика: распределение NER-сущностей по типам")
print("=" * 60)

stats = (session.query(Entity.ner_type, func.count(Entity.id))
         .group_by(Entity.ner_type)
         .order_by(func.count(Entity.id).desc())
         .all())

for ner_type, count in stats:
    print(f"  {ner_type:10s}: {count} шт.")

# --- 5. Полнотекстовый поиск (имитация для SQLite) ---
print("\n" + "=" * 60)
print("5. Поиск документов по ключевым словам (имитация FTS)")
print("=" * 60)

query_term = 'банкротство'
found = (session.query(Document)
         .filter(Document.text.ilike(f'%{query_term}%'))
         .all())

for d in found:
    # Формируем сниппет вручную (в PostgreSQL — ts_headline)
    idx = d.text.lower().find(query_term)
    start = max(0, idx - 40)
    end = min(len(d.text), idx + len(query_term) + 40)
    snippet = '...' + d.text[start:end] + '...'
    print(f"  Документ ID={d.id}: {snippet}")

print("\nПримечание: в PostgreSQL используется ts_headline() для подсветки терминов.")
print("SQL-эквивалент:")
print("""  SELECT title, ts_headline('russian', text, query) AS snippet
  FROM documents, to_tsquery('russian', 'банкротство') query
  WHERE to_tsvector('russian', text) @@ query
  ORDER BY ts_rank(to_tsvector('russian', text), query) DESC;""")

## 5. Обеспечение целостности и безопасности данных

Обеспечение целостности и безопасности данных является одним из фундаментальных требований к любой информационной системе, работающей с юридической информацией. Утрата, искажение или несанкционированный доступ к судебным актам, персональным данным участников процесса или результатам аналитической обработки может повлечь серьёзные правовые последствия для оператора системы. В данном разделе рассматриваются механизмы обеспечения целостности, резервного копирования, управления доступом и защиты данных на уровне СУБД.

### 5.1. Ограничения целостности и триггеры

Ограничения целостности (integrity constraints) представляют собой декларативные правила, которым должны удовлетворять данные в базе. PostgreSQL поддерживает несколько типов ограничений, каждый из которых защищает от определённого класса ошибок. Ограничение $\texttt{NOT NULL}$ гарантирует обязательность заполнения поля. Ограничение $\texttt{UNIQUE}$ исключает дублирование значений в указанном столбце или группе столбцов, что критически важно для полей вроде номера дела ($\texttt{cases.number}$). Ограничение $\texttt{CHECK}$ задаёт произвольное логическое условие, которому должно удовлетворять значение столбца: например, $\texttt{CHECK (confidence BETWEEN 0.0 AND 1.0)}$ гарантирует, что оценка уверенности NER-модели находится в допустимом диапазоне. Ограничение внешнего ключа ($\texttt{FOREIGN KEY}$) обеспечивает ссылочную целостность, исключая ситуации, когда документ ссылается на несуществующее дело.

Триггеры (triggers) обеспечивают автоматическое выполнение произвольных действий при наступлении определённых событий (вставка, обновление, удаление записей). В юридической базе данных триггеры могут использоваться для автоматического ведения журнала изменений (audit log), валидации сложных бизнес-правил, которые невозможно выразить декларативными ограничениями, а также для автоматического обновления производных данных. Например, триггер $\texttt{AFTER INSERT}$ на таблице $\texttt{documents}$ может автоматически инициировать обновление полнотекстового индекса или запуск конвейера NER-обработки.

Формально триггер определяется как пара $\langle E, A \rangle$, где $E$ — событие (INSERT, UPDATE, DELETE на определённой таблице), а $A$ — действие (функция, выполняемая при наступлении события). Триггеры могут быть определены как $\texttt{BEFORE}$ (выполняются до основной операции и могут её отменить или модифицировать) или $\texttt{AFTER}$ (выполняются после успешного завершения основной операции). В контексте аудита юридических данных триггеры $\texttt{AFTER}$ являются предпочтительными, поскольку фиксируют только фактически выполненные изменения.

### 5.2. Резервное копирование и восстановление

Стратегия резервного копирования юридической базы данных должна обеспечивать возможность восстановления данных до любого момента времени (Point-in-Time Recovery, PITR) с минимальной потерей данных. PostgreSQL предоставляет несколько уровней резервного копирования.

Логическое резервное копирование с помощью утилиты $\texttt{pg\_dump}$ создаёт текстовый файл с SQL-командами, воспроизводящими структуру и содержимое базы данных. Данный метод обеспечивает полную переносимость между версиями PostgreSQL и позволяет восстанавливать отдельные таблицы, однако для больших баз время выполнения может быть значительным.

Физическое резервное копирование ($\texttt{pg\_basebackup}$) создаёт копию всех файлов данных на уровне файловой системы, что обеспечивает более высокую производительность для больших баз. В сочетании с архивированием журналов записи предварительной записи (Write-Ahead Log, WAL) физическое копирование обеспечивает PITR: база данных может быть восстановлена до состояния на любой момент времени, покрытый сохранёнными WAL-сегментами.

Для промышленных юридических систем рекомендуется комбинированная стратегия: ежедневное полное физическое копирование с непрерывным архивированием WAL-журналов. Параметр Recovery Point Objective (RPO) — максимально допустимый объём потерянных данных — для юридических систем, как правило, не должен превышать нескольких минут.

### 5.3. Управление доступом

Модель управления доступом в PostgreSQL основана на понятиях ролей (roles) и привилегий (privileges). Роль — именованная сущность, которой могут быть назначены права на выполнение операций с объектами базы данных. Роли могут наследовать привилегии других ролей, образуя иерархию доступа.

В юридической информационной системе типичная структура ролей включает несколько уровней. Роль администратора обладает полными правами на управление структурой и данными. Роль аналитика имеет права на чтение всех таблиц и выполнение аналитических запросов, но не имеет прав на модификацию данных. Роль приложения обладает правами на чтение и запись данных в пределах определённых таблиц, необходимых для функционирования автоматизированных конвейеров. Роль аудитора имеет доступ только для чтения журналов аудита.

Принцип наименьших привилегий (Principle of Least Privilege) требует, чтобы каждая роль обладала минимальным набором прав, необходимым для выполнения своих функций. Применение данного принципа существенно снижает риск несанкционированного доступа или случайного повреждения данных.

### 5.4. Защита данных

Защита данных в юридической базе охватывает несколько уровней. Шифрование данных при передаче обеспечивается протоколом TLS (Transport Layer Security) для всех соединений между клиентами и сервером СУБД. Шифрование данных при хранении (encryption at rest) может быть реализовано на уровне файловой системы или посредством расширений PostgreSQL.

Особого внимания требует обработка персональных данных участников судебных процессов в соответствии с требованиями Федерального закона № 152-ФЗ «О персональных данных». Система должна обеспечивать возможность обезличивания персональных данных, ограничения доступа к чувствительным полям на уровне строк (Row-Level Security, RLS) и ведения журнала доступа к персональным данным.

---


In [ ]:
# ============================================================
# Демонстрация: SQL-скрипты для обеспечения безопасности
# Триггер аудита, роли, привилегии
# ============================================================

security_sql = """
-- ============================================================
-- 1. Таблица аудита изменений документов
-- ============================================================
CREATE TABLE audit_log (
    id          SERIAL PRIMARY KEY,
    table_name  VARCHAR(50)  NOT NULL,
    record_id   INTEGER      NOT NULL,
    operation   VARCHAR(10)  NOT NULL,   -- INSERT / UPDATE / DELETE
    old_data    JSONB,                    -- предыдущее состояние записи
    new_data    JSONB,                    -- новое состояние записи
    changed_by  VARCHAR(100) DEFAULT current_user,
    changed_at  TIMESTAMP    DEFAULT CURRENT_TIMESTAMP
);

-- ============================================================
-- 2. Триггерная функция для журналирования изменений
-- ============================================================
CREATE OR REPLACE FUNCTION fn_audit_documents()
RETURNS TRIGGER AS $$
BEGIN
    IF TG_OP = 'INSERT' THEN
        INSERT INTO audit_log(table_name, record_id, operation, new_data)
        VALUES (TG_TABLE_NAME, NEW.id, 'INSERT', to_jsonb(NEW));
        RETURN NEW;
    ELSIF TG_OP = 'UPDATE' THEN
        INSERT INTO audit_log(table_name, record_id, operation, old_data, new_data)
        VALUES (TG_TABLE_NAME, NEW.id, 'UPDATE', to_jsonb(OLD), to_jsonb(NEW));
        RETURN NEW;
    ELSIF TG_OP = 'DELETE' THEN
        INSERT INTO audit_log(table_name, record_id, operation, old_data)
        VALUES (TG_TABLE_NAME, OLD.id, 'DELETE', to_jsonb(OLD));
        RETURN OLD;
    END IF;
END;
$$ LANGUAGE plpgsql;

-- Привязка триггера к таблице documents
CREATE TRIGGER trg_audit_documents
    AFTER INSERT OR UPDATE OR DELETE ON documents
    FOR EACH ROW EXECUTE FUNCTION fn_audit_documents();

-- ============================================================
-- 3. Управление доступом: роли и привилегии
-- ============================================================

-- Роль аналитика: только чтение
CREATE ROLE analyst WITH LOGIN PASSWORD 'secure_password_1';
GRANT SELECT ON ALL TABLES IN SCHEMA public TO analyst;

-- Роль приложения: чтение + запись в рабочие таблицы
CREATE ROLE app_service WITH LOGIN PASSWORD 'secure_password_2';
GRANT SELECT, INSERT, UPDATE ON documents, entities, annotations TO app_service;
GRANT USAGE, SELECT ON ALL SEQUENCES IN SCHEMA public TO app_service;

-- Роль аудитора: только журнал аудита
CREATE ROLE auditor WITH LOGIN PASSWORD 'secure_password_3';
GRANT SELECT ON audit_log TO auditor;

-- ============================================================
-- 4. Row-Level Security для ограничения доступа по роли
-- ============================================================
ALTER TABLE documents ENABLE ROW LEVEL SECURITY;

-- Аналитик видит только завершённые дела
CREATE POLICY analyst_docs_policy ON documents
    FOR SELECT TO analyst
    USING (case_id IN (
        SELECT id FROM cases WHERE status = 'завершено'
    ));

-- Приложение имеет полный доступ
CREATE POLICY app_full_access ON documents
    FOR ALL TO app_service
    USING (true) WITH CHECK (true);
"""

print("SQL-скрипт обеспечения безопасности юридической БД")
print("=" * 60)
print(security_sql)
print("\nОсновные механизмы защиты:")
print("  1. Триггер аудита — журналирует все изменения в documents")
print("  2. Ролевая модель — analyst / app_service / auditor")
print("  3. Row-Level Security — ограничение доступа на уровне строк")

## 6. Масштабирование и оптимизация базы данных

По мере роста объёма хранимых данных и числа одновременных пользователей производительность базы данных неизбежно становится критическим фактором работоспособности системы. Проблема масштабирования приобретает особую остроту для юридических информационно-поисковых систем, где кумулятивный характер данных (судебные акты не удаляются, а лишь накапливаются) приводит к монотонному росту объёма базы.

### 6.1. Вертикальное и горизонтальное масштабирование

Вертикальное масштабирование (scale-up) предполагает увеличение вычислительных ресурсов одного сервера: оперативной памяти, числа ядер процессора, скорости и объёма дисковой подсистемы. Данный подход прост в реализации и не требует изменения архитектуры приложения, однако имеет физический предел, определяемый характеристиками доступного аппаратного обеспечения, а также нелинейный рост стоимости ресурсов.

Горизонтальное масштабирование (scale-out) предполагает распределение нагрузки между несколькими серверами. В контексте реляционных СУБД горизонтальное масштабирование реализуется посредством двух основных механизмов: репликации и шардинга.

Репликация создаёт копии базы данных на нескольких серверах. Различают синхронную репликацию, при которой транзакция считается завершённой только после подтверждения записи на всех репликах (обеспечивает строгую согласованность, но увеличивает задержку), и асинхронную репликацию, при которой данные копируются с допустимой задержкой (обеспечивает более высокую производительность записи ценой возможной временной несогласованности). В PostgreSQL потоковая репликация (streaming replication) обеспечивает передачу WAL-записей от первичного сервера к репликам в режиме, близком к реальному времени. Типичная схема включает один первичный сервер для операций записи и несколько реплик для операций чтения, что позволяет горизонтально масштабировать производительность поисковых запросов.

Шардинг (partitioning across nodes) предполагает разбиение данных на фрагменты (шарды), каждый из которых размещается на отдельном сервере. Критерий разбиения определяется характером запросов: для юридической базы данных естественным критерием является юрисдикция (регион суда) или временной диапазон (год вынесения решения). PostgreSQL не предоставляет встроенного механизма шардинга, однако расширение Citus реализует прозрачное распределение данных и параллельное выполнение запросов.

### 6.2. Оптимизация производительности

Оптимизация производительности базы данных — непрерывный процесс, основанный на мониторинге, диагностике и итеративном улучшении конфигурации и структуры запросов.

Анализ планов выполнения запросов с помощью $\texttt{EXPLAIN ANALYZE}$ позволяет выявить неэффективные операции: полное сканирование таблиц (Sequential Scan) там, где ожидается использование индекса, избыточные сортировки, неоптимальный порядок соединения таблиц. Стоимость плана выражается в условных единицах, пропорциональных числу операций ввода-вывода и вычислительных операций. Оптимизатор PostgreSQL использует статистику о распределении данных (собираемую командой $\texttt{ANALYZE}$) для выбора наиболее эффективного плана.

Кэширование часто запрашиваемых данных снижает нагрузку на дисковую подсистему. PostgreSQL поддерживает настраиваемый буферный пул ($\texttt{shared\_buffers}$), а на уровне приложения может использоваться внешний кэш (Redis, Memcached) для хранения результатов частых запросов. Параметр $\texttt{effective\_cache\_size}$ сообщает планировщику запросов об ожидаемом объёме данных, доступных в кэше операционной системы, что влияет на выбор между последовательным сканированием и индексным доступом.

Материализованные представления (materialized views) обеспечивают предварительное вычисление и хранение результатов сложных аналитических запросов. В юридической системе типичным примером является агрегированная статистика по категориям дел, распределение документов по судам и временным периодам. Команда $\texttt{REFRESH MATERIALIZED VIEW}$ обновляет содержимое представления, что может выполняться по расписанию (например, ежедневно).

---


In [ ]:
# ============================================================
# Демонстрация: анализ производительности запросов
# и создание материализованного представления
# ============================================================

# --- 1. EXPLAIN ANALYZE (SQL-пример для PostgreSQL) ---
explain_example = """
-- Анализ плана выполнения запроса полнотекстового поиска
EXPLAIN ANALYZE
SELECT d.title, c.number,
       ts_rank(to_tsvector('russian', d.text), query) AS rank
FROM documents d
JOIN cases c ON d.case_id = c.id,
     to_tsquery('russian', 'банкротство & субсидиарная') query
WHERE to_tsvector('russian', d.text) @@ query
ORDER BY rank DESC
LIMIT 20;

-- Типичный вывод:
-- Limit  (cost=28.48..28.49 rows=4 width=...)
--   -> Sort  (cost=28.48..28.49 rows=4 ...)
--        Sort Key: ts_rank(...) DESC
--        -> Bitmap Heap Scan on documents d  (cost=12.08..28.45 ...)
--              Recheck Cond: (to_tsvector('russian', text) @@ ...)
--              -> Bitmap Index Scan on idx_documents_fts  (cost=0.00..12.08 ...)
-- Planning Time: 0.852 ms
-- Execution Time: 1.234 ms
"""

print("1. Пример анализа плана выполнения запроса (EXPLAIN ANALYZE)")
print("=" * 60)
print(explain_example)

# --- 2. Материализованное представление ---
matview_sql = """
-- Материализованное представление: статистика по категориям дел
CREATE MATERIALIZED VIEW mv_case_statistics AS
SELECT
    c.category,
    ct.level         AS court_level,
    EXTRACT(YEAR FROM c.date_filed) AS year,
    COUNT(*)         AS total_cases,
    COUNT(DISTINCT c.court_id) AS courts_involved,
    COUNT(d.id)      AS total_documents
FROM cases c
JOIN courts ct ON c.court_id = ct.id
LEFT JOIN documents d ON d.case_id = c.id
GROUP BY c.category, ct.level, EXTRACT(YEAR FROM c.date_filed)
ORDER BY year DESC, total_cases DESC;

-- Индекс для быстрого доступа к представлению
CREATE INDEX idx_mv_stats_category
    ON mv_case_statistics(category);

-- Обновление (по расписанию cron или pg_cron)
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_case_statistics;
"""

print("\n2. Материализованное представление для аналитики")
print("=" * 60)
print(matview_sql)

# --- 3. Демонстрация сравнения планов на SQLite ---
import time

print("\n3. Сравнение производительности: LIKE vs индексированный поиск")
print("=" * 60)

# Добавим ещё документов для эксперимента
for i in range(500):
    session.add(Document(
        title=f'Тестовый документ #{i+2}',
        date=date(2024, 1, 1),
        text=f'Содержание тестового документа номер {i+2}. '
             f'{"Банкротство организации. " if i % 5 == 0 else ""}'
             f'Рассмотрение дела в арбитражном суде.',
        doc_type='определение',
        case_id=case1.id
    ))
session.commit()

total_docs = session.query(func.count(Document.id)).scalar()
print(f"  Всего документов в базе: {total_docs}")

# Поиск методом LIKE (аналог Sequential Scan)
start = time.perf_counter()
for _ in range(100):
    results = (session.query(Document)
               .filter(Document.text.ilike('%банкротство%'))
               .all())
elapsed_like = (time.perf_counter() - start) / 100

print(f"  LIKE-поиск:        {elapsed_like*1000:.2f} мс (в среднем за 100 итераций)")
print(f"  Найдено документов: {len(results)}")
print("\n  В PostgreSQL с GIN-индексом полнотекстовый поиск")
print("  выполняется на 1-2 порядка быстрее на больших объёмах данных.")

## 7. Работа с большими объёмами данных

Юридические информационные системы характеризуются кумулятивным характером накопления данных: судебные акты, нормативные документы и материалы дел не удаляются из системы, а сохраняются в течение длительного времени (нередко — бессрочно). С учётом ежегодного прироста в несколько миллионов документов объём базы данных в крупной юридической системе может достигать десятков и сотен терабайт в течение нескольких лет эксплуатации. Работа с такими объёмами требует специальных архитектурных решений.

### 7.1. Архивация данных

Архивация предполагает перенос устаревших или неактуальных данных из основного хранилища в архивное, обладающее более низкой стоимостью за единицу объёма, но менее высокой производительностью доступа. В юридическом контексте архивации подлежат документы по делам, завершённым более определённого срока назад (например, более 5 лет), при условии сохранения возможности доступа к ним при необходимости.

Архитектурно архивная система может быть реализована как отдельная база данных на менее производительном (и менее дорогостоящем) оборудовании, облачное объектное хранилище (Amazon S3, MinIO) для хранения полных текстов документов, или отдельная партиция основной базы данных с пониженными параметрами производительности.

Процесс архивации должен быть автоматизирован и включать верификацию целостности перенесённых данных, обновление метаданных (пометку о переносе в архив) и опциональное удаление архивированных записей из основной базы. При поступлении запроса к архивированному документу система выполняет прозрачный доступ к архивному хранилищу, что может требовать дополнительного времени.

### 7.2. Партиционирование таблиц

Партиционирование (секционирование) — разделение одной логической таблицы на несколько физических секций, каждая из которых хранит подмножество данных, определяемое значением ключа партиционирования. PostgreSQL поддерживает декларативное партиционирование по диапазону ($\texttt{RANGE}$), по списку ($\texttt{LIST}$) и по хешу ($\texttt{HASH}$).

Для юридической базы данных наиболее естественным является партиционирование по диапазону дат: каждая партиция содержит документы, относящиеся к определённому временному периоду (кварталу, году). Такое разбиение обеспечивает три существенных преимущества. Во-первых, запросы, фильтрующие данные по дате (что является типичным для юридического анализа), обращаются только к релевантным партициям, что значительно сокращает объём сканируемых данных — данный механизм называется отсечением партиций (partition pruning). Во-вторых, архивация устаревших данных сводится к отсоединению (detach) целой партиции, что является мгновенной операцией в отличие от построчного удаления. В-третьих, обслуживание (индексирование, вакуумирование) может выполняться для каждой партиции независимо, что снижает влияние на производительность рабочей нагрузки.

Эффективность партиционирования можно оценить количественно. Пусть таблица содержит $N$ записей, распределённых по $P$ партициям приблизительно равномерно. Типичный запрос с фильтром по дате обращается к $k$ партициям (где $k \ll P$). В этом случае объём сканируемых данных составляет $\frac{k \cdot N}{P}$ вместо $N$, что обеспечивает ускорение в $\frac{P}{k}$ раз при условии отсечения партиций.

---


In [ ]:
# ============================================================
# Демонстрация: SQL-скрипт партиционирования таблицы документов
# и моделирование эффекта отсечения партиций
# ============================================================

partition_sql = """
-- ============================================================
-- Партиционирование таблицы documents по дате (по годам)
-- ============================================================

-- Родительская таблица (партиционированная)
CREATE TABLE documents_partitioned (
    id          SERIAL,
    title       VARCHAR(500) NOT NULL,
    date        DATE         NOT NULL,
    text        TEXT         NOT NULL,
    doc_type    VARCHAR(50)  NOT NULL,
    case_id     INTEGER      NOT NULL,
    created_at  TIMESTAMP    DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (id, date)     -- ключ партиционирования входит в PK
) PARTITION BY RANGE (date);

-- Создание партиций по годам
CREATE TABLE docs_2020 PARTITION OF documents_partitioned
    FOR VALUES FROM ('2020-01-01') TO ('2021-01-01');

CREATE TABLE docs_2021 PARTITION OF documents_partitioned
    FOR VALUES FROM ('2021-01-01') TO ('2022-01-01');

CREATE TABLE docs_2022 PARTITION OF documents_partitioned
    FOR VALUES FROM ('2022-01-01') TO ('2023-01-01');

CREATE TABLE docs_2023 PARTITION OF documents_partitioned
    FOR VALUES FROM ('2023-01-01') TO ('2024-01-01');

CREATE TABLE docs_2024 PARTITION OF documents_partitioned
    FOR VALUES FROM ('2024-01-01') TO ('2025-01-01');

CREATE TABLE docs_default PARTITION OF documents_partitioned DEFAULT;

-- Индексы создаются на каждой партиции автоматически
CREATE INDEX idx_docs_part_date ON documents_partitioned(date);
CREATE INDEX idx_docs_part_type ON documents_partitioned(doc_type);

-- Полнотекстовый индекс наследуется партициями
CREATE INDEX idx_docs_part_fts
    ON documents_partitioned
    USING gin(to_tsvector('russian', text));

-- Запрос с отсечением партиций
-- Планировщик обращается ТОЛЬКО к docs_2024
EXPLAIN (COSTS OFF)
SELECT * FROM documents_partitioned
WHERE date BETWEEN '2024-01-01' AND '2024-12-31'
  AND doc_type = 'решение';
"""

print("SQL-скрипт партиционирования документов по годам")
print("=" * 60)
print(partition_sql)

# --- Моделирование эффекта партиционирования ---
import matplotlib.pyplot as plt
import numpy as np

years = 5
total_docs = 5_000_000  # общее число документов
docs_per_year = total_docs // years

# Время запроса: пропорционально объёму сканируемых данных
time_full_scan = total_docs / 1e6  # условные единицы
time_partitioned = docs_per_year / 1e6

fig, ax = plt.subplots(figsize=(10, 5))
methods = ['Без партиционирования\n(полное сканирование)', 'С партиционированием\n(1 партиция = 1 год)']
times = [time_full_scan, time_partitioned]
colors = ['#EF5350', '#66BB6A']

bars = ax.bar(methods, times, color=colors, edgecolor='#333', width=0.5)
for bar, t in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{t:.1f} усл. ед.', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('Относительное время запроса', fontsize=12)
ax.set_title(f'Эффект партиционирования ({total_docs:,} документов, {years} партиций)',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, max(times) * 1.3)
plt.tight_layout()
plt.show()

speedup = time_full_scan / time_partitioned
print(f"\nУскорение при запросе к одному году: {speedup:.0f}x")
print(f"  Без партиционирования: сканирование {total_docs:,} записей")
print(f"  С партиционированием:  сканирование {docs_per_year:,} записей")

## 8. Практические рекомендации по эксплуатации базы данных

Проектирование и первоначальная реализация базы данных составляют лишь начальный этап жизненного цикла информационной системы. Основная часть затрат приходится на эксплуатационное сопровождение, включающее мониторинг производительности, обновление схемы, устранение инцидентов и адаптацию к изменяющимся требованиям. Настоящий раздел систематизирует рекомендации по организации эксплуатации юридической базы данных.

### 8.1. Документирование базы данных

Полнота и актуальность документации базы данных является необходимым условием её сопровождаемости. Документация должна включать описание концептуальной модели (ER-диаграмма с пояснениями к каждой сущности и связи), спецификацию физической схемы (таблицы, столбцы, типы данных, ограничения, индексы), описание процедур миграции и обновления схемы, а также руководство администратора с описанием процедур резервного копирования, восстановления и мониторинга.

Для визуализации структуры базы данных применяются специализированные инструменты: pgAdmin (встроенный веб-интерфейс для PostgreSQL), DBeaver (универсальный клиент для работы с различными СУБД), а также средства автоматической генерации документации из метаданных базы (SchemaSpy, dbdocs.io). Документация должна версионироваться совместно с исходным кодом приложения и обновляться при каждом изменении схемы.

В академическом контексте ведение подробной документации является не только инженерной необходимостью, но и требованием научной воспроизводимости. Исследователь, работающий с базой данных судебных актов, должен иметь возможность точно воспроизвести процедуру формирования выборки, описать использованные фильтры и критерии отбора. Документация базы данных обеспечивает эту возможность.

### 8.2. Мониторинг и аудит

Мониторинг производительности базы данных позволяет своевременно выявлять деградацию и предотвращать критические сбои. PostgreSQL предоставляет системное представление $\texttt{pg\_stat\_statements}$ для анализа наиболее ресурсоёмких запросов, $\texttt{pg\_stat\_user\_tables}$ для отслеживания статистики обращений к таблицам, $\texttt{pg\_stat\_bgwriter}$ для оценки эффективности буферного пула.

Ключевые метрики, подлежащие мониторингу, включают среднее время выполнения запросов (по типам: SELECT, INSERT, UPDATE), процент использования кэша (cache hit ratio), объём дискового пространства и темп его роста, число активных соединений и очередь ожидания блокировок, а также частоту выполнения автовакуума. Для визуализации метрик и настройки оповещений применяются системы мониторинга Prometheus + Grafana с экспортёром $\texttt{postgres\_exporter}$.

Аудит действий пользователей обеспечивается расширением $\texttt{pgAudit}$, которое фиксирует в журнале все SQL-операции с детализацией по пользователям и объектам. Аудит является обязательным требованием для систем, обрабатывающих персональные данные в соответствии с законодательством Российской Федерации.

### 8.3. Миграция и обновление схемы

Эволюция схемы базы данных представляет собой неизбежный процесс, обусловленный изменением требований, добавлением новых функций или исправлением ошибок проектирования. Для управления миграциями используется инструмент Alembic, интегрированный с SQLAlchemy. Alembic поддерживает версионирование миграций, автоматическую генерацию скриптов миграции на основе сравнения модели ORM с текущим состоянием базы, а также откат миграций в случае обнаружения проблем.

Каждая миграция представляет собой пару функций: $\texttt{upgrade()}$ (применение изменений) и $\texttt{downgrade()}$ (откат изменений). Миграции версионируются и хранятся в системе контроля версий совместно с исходным кодом приложения, что обеспечивает синхронизацию версий кода и схемы базы данных. При развёртывании в промышленной среде миграции выполняются в рамках транзакций, что гарантирует атомарность обновления.

---


In [ ]:
# ============================================================
# Демонстрация: структура миграции Alembic
# Добавление столбца и нового индекса
# ============================================================

alembic_migration = '''
# Миграция: добавление столбца summary в таблицу documents
# Revision ID: a1b2c3d4e5f6
# Create Date:  2024-09-15

from alembic import op
import sqlalchemy as sa

revision = 'a1b2c3d4e5f6'
down_revision = None

def upgrade():
    # 1. Столбец для автоматического резюме
    op.add_column('documents', sa.Column(
        'summary', sa.Text(), nullable=True))
    # 2. Столбец для эмбеддинга (pgvector: VECTOR(768))
    op.add_column('documents', sa.Column(
        'embedding', sa.Text(), nullable=True))
    # 3. Составной индекс
    op.create_index('idx_docs_date_type', 'documents',
                    ['date', 'doc_type'], unique=False)

def downgrade():
    op.drop_index('idx_docs_date_type', table_name='documents')
    op.drop_column('documents', 'embedding')
    op.drop_column('documents', 'summary')
'''

print("Пример миграции Alembic для юридической БД")
print("=" * 60)
print(alembic_migration)

print("\nТиповые команды Alembic:")
print("=" * 60)
commands = [
    ("alembic init migrations",         "Инициализация проекта миграций"),
    ("alembic revision --autogenerate",  "Автогенерация миграции"),
    ("alembic upgrade head",             "Применение всех новых миграций"),
    ("alembic downgrade -1",             "Откат последней миграции"),
    ("alembic history",                  "Просмотр истории миграций"),
]
for cmd, desc in commands:
    print(f"  {cmd}")
    print(f"    -> {desc}")

## 9. Практические примеры реализации

Настоящий раздел демонстрирует комплексный пример взаимодействия с юридической базой данных через программный интерфейс (API). В промышленных системах прямой доступ пользователей к SQL-интерфейсу базы данных, как правило, не допускается по соображениям безопасности и удобства. Вместо этого реализуется серверное приложение (backend), предоставляющее REST API для выполнения операций поиска, добавления и обновления данных. Клиентские приложения (веб-интерфейс, мобильные приложения, внешние системы) взаимодействуют с базой данных исключительно через этот промежуточный слой.

### 9.1. REST API с использованием FastAPI

Фреймворк FastAPI обеспечивает высокопроизводительную реализацию REST API на языке Python с автоматической валидацией входных данных, сериализацией ответов и генерацией интерактивной документации (Swagger UI). Интеграция FastAPI с SQLAlchemy ORM позволяет реализовать паттерн «репозиторий» (Repository pattern), инкапсулирующий логику доступа к данным и обеспечивающий разделение ответственности между компонентами приложения.

Типичная архитектура серверного приложения включает три слоя. Слой маршрутизации (router) определяет HTTP-эндпойнты и их параметры. Сервисный слой (service) реализует бизнес-логику операций. Слой доступа к данным (repository) выполняет запросы к базе данных через ORM. Такое разделение обеспечивает тестируемость каждого слоя независимо от остальных, а также упрощает замену компонентов: например, переход с PostgreSQL на другую СУБД требует модификации только слоя доступа к данным.

### 9.2. Пример полного конвейера

Ниже представлен комплексный пример, демонстрирующий полный цикл работы с юридической базой данных: от определения Pydantic-схем для валидации данных до реализации API-эндпойнтов с пагинацией и фильтрацией.

---


In [ ]:
# ============================================================
# Демонстрация: REST API для юридической базы данных
# FastAPI + SQLAlchemy ORM (структурный пример)
# ============================================================

api_structure = """
# == app/schemas.py ==
from pydantic import BaseModel, Field
from datetime import date
from typing import Optional, List

class DocumentBase(BaseModel):
    title: str = Field(..., max_length=500)
    date: date
    doc_type: str
    case_id: int

class DocumentCreate(DocumentBase):
    text: str = Field(..., min_length=10)

class DocumentResponse(DocumentBase):
    id: int
    text: str
    entity_count: Optional[int] = 0
    class Config:
        from_attributes = True

class SearchResult(BaseModel):
    document_id: int
    title: str
    snippet: str
    rank: float = Field(..., ge=0)


# == app/repository.py ==
from sqlalchemy.orm import Session
from sqlalchemy import text

class DocumentRepository:
    def __init__(self, db: Session):
        self.db = db

    def get_by_id(self, doc_id: int):
        return self.db.query(Document).filter(Document.id == doc_id).first()

    def list_documents(self, skip=0, limit=20, doc_type=None):
        q = self.db.query(Document)
        if doc_type:
            q = q.filter(Document.doc_type == doc_type)
        return q.order_by(Document.date.desc()).offset(skip).limit(limit).all()

    def fulltext_search(self, query_text, limit=20):
        sql = text("""
            SELECT d.id, d.title,
                   ts_headline('russian', d.text, query,
                               'MaxWords=30, MinWords=15') AS snippet,
                   ts_rank(to_tsvector('russian', d.text), query) AS rank
            FROM documents d,
                 plainto_tsquery('russian', :q) query
            WHERE to_tsvector('russian', d.text) @@ query
            ORDER BY rank DESC LIMIT :lim
        """)
        return self.db.execute(sql, {"q": query_text, "lim": limit}).fetchall()


# == app/routes.py ==
from fastapi import APIRouter, Depends, Query, HTTPException

router = APIRouter(prefix="/api/v1", tags=["documents"])

@router.get("/documents", response_model=List[DocumentResponse])
def list_documents(skip: int = 0, limit: int = 20,
                   doc_type: str = None, db: Session = Depends(get_db)):
    repo = DocumentRepository(db)
    return repo.list_documents(skip=skip, limit=limit, doc_type=doc_type)

@router.get("/search", response_model=List[SearchResult])
def search_documents(q: str = Query(..., min_length=2),
                     limit: int = 20, db: Session = Depends(get_db)):
    repo = DocumentRepository(db)
    results = repo.fulltext_search(q, limit=limit)
    return [SearchResult(document_id=r.id, title=r.title,
                         snippet=r.snippet, rank=r.rank) for r in results]

@router.post("/documents", response_model=DocumentResponse, status_code=201)
def create_document(doc: DocumentCreate, db: Session = Depends(get_db)):
    repo = DocumentRepository(db)
    return repo.create(doc.model_dump())
"""

print("Структурный пример REST API юридической ИПС")
print("=" * 60)
print(api_structure)
print("API-эндпойнты:")
print("  GET  /api/v1/documents      -- список с пагинацией и фильтрами")
print("  GET  /api/v1/search?q=...   -- полнотекстовый поиск")
print("  POST /api/v1/documents      -- добавление документа")


## 10. Интеграция базы данных с NLP-конвейером

Юридическая база данных не является изолированным компонентом: она тесно интегрирована с конвейером обработки естественного языка, рассмотренным в предыдущих лекциях курса. Эта интеграция обеспечивает автоматическое обогащение хранимых документов аналитическими метаданными и создаёт основу для интеллектуального поиска.

### 10.1. Архитектура интеграции

Типичный конвейер обработки нового документа включает несколько последовательных стадий. На первой стадии документ поступает в систему (через API или пакетную загрузку) и сохраняется в таблице $\texttt{documents}$ с базовыми метаданными. На второй стадии запускается NER-модель, извлекающая из текста именованные сущности (лица, организации, даты, ссылки на законы, суммы), которые сохраняются в таблице $\texttt{entities}$. На третьей стадии классификатор присваивает документу тематические метки (банкротство, трудовые споры, налоговые дела), записываемые в таблицу $\texttt{annotations}$. На четвёртой стадии модель суммаризации формирует краткое резюме документа, сохраняемое в столбце $\texttt{summary}$. На пятой стадии вычисляется векторное представление (эмбеддинг) документа, используемое для семантического поиска.

Данный конвейер может быть реализован как синхронно (в рамках обработки HTTP-запроса), так и асинхронно (через очередь задач). Для промышленных систем предпочтителен асинхронный подход с использованием брокера сообщений (RabbitMQ, Kafka), поскольку NLP-обработка является вычислительно затратной операцией и её синхронное выполнение привело бы к неприемлемому времени отклика API.

### 10.2. Хранение и использование векторных представлений

Современные методы семантического поиска основаны на использовании плотных векторных представлений (dense embeddings) документов и запросов. Документ $d$ отображается в вектор $\mathbf{v}_d \in \mathbb{R}^n$ (где $n$ -- размерность модели, типично $n = 768$ для моделей семейства BERT), а запрос $q$ -- в вектор $\mathbf{v}_q$ той же размерности. Релевантность оценивается косинусным сходством:

$$\text{sim}(q, d) = \frac{\mathbf{v}_q \cdot \mathbf{v}_d}{\|\mathbf{v}_q\| \cdot \|\mathbf{v}_d\|}$$

Расширение pgvector для PostgreSQL обеспечивает хранение векторов и выполнение операций приближённого поиска ближайших соседей непосредственно в базе данных. Поддерживаются два типа индексов: IVFFlat (инвертированный файловый индекс с плоской квантизацией) и HNSW (иерархический граф близости). HNSW-индекс обеспечивает более высокое качество поиска (recall) при сопоставимой производительности и является рекомендуемым выбором для большинства приложений.

---


In [ ]:
# ============================================================
# Демонстрация: интеграция NLP-конвейера с базой данных
# Автоматическое обогащение документа метаданными
# ============================================================
import re
from collections import Counter

def simple_ner(text):
    """Упрощённое извлечение именованных сущностей из юридического текста."""
    entities = []
    # Ссылки на статьи законов
    for m in re.finditer(r'стать[яиеюёй]+\s+[\d.]+\s+[А-Яа-яё\s]+(?:закона|кодекса|ФЗ)', text):
        entities.append({'name': m.group().strip(), 'type': 'LAW_REF',
                         'start': m.start(), 'end': m.end()})
    # Денежные суммы
    for m in re.finditer(r'[\d\s]+(?:рубл|руб\\.)', text):
        entities.append({'name': m.group().strip(), 'type': 'MONEY',
                         'start': m.start(), 'end': m.end()})
    # Организации
    for m in re.finditer(r'(?:ООО|ПАО|АО|ЗАО)\s*[\xab\"][^\xbb\"]+[\xbb\"]', text):
        entities.append({'name': m.group(), 'type': 'ORG',
                         'start': m.start(), 'end': m.end()})
    return entities

def simple_classify(text):
    """Классификация документа по ключевым словам."""
    keywords = {
        'банкротство': ['банкротств', 'конкурсн', 'несостоятельност', 'кредитор'],
        'трудовые споры': ['трудов', 'увольнен', 'работник', 'работодател'],
        'налоговые дела': ['налог', 'ФНС', 'налогоплательщик', 'декларац'],
        'корпоративные споры': ['акционер', 'долев', 'устав', 'дивиденд'],
    }
    text_lower = text.lower()
    scores = {}
    for label, kws in keywords.items():
        scores[label] = sum(1 for kw in kws if kw.lower() in text_lower)
    if max(scores.values()) > 0:
        best = max(scores, key=scores.get)
        return best, min(scores[best] / len(keywords[best]), 1.0)
    return 'прочее', 0.5

def process_document(doc_obj, session_obj):
    """Полный конвейер обработки нового документа."""
    print(f"\nОбработка документа ID={doc_obj.id}: '{doc_obj.title[:50]}...'")
    print("-" * 60)

    # Шаг 1: NER
    ner_results = simple_ner(doc_obj.text)
    print(f"  [NER] Извлечено сущностей: {len(ner_results)}")
    for ent in ner_results:
        entity_obj = Entity(
            document_id=doc_obj.id, name=ent['name'],
            ner_type=ent['type'], start_pos=ent['start'],
            end_pos=ent['end'], confidence=0.85)
        session_obj.add(entity_obj)
        print(f"    [{ent['type']:10s}] {ent['name'][:50]}")

    # Шаг 2: Классификация
    label, conf = simple_classify(doc_obj.text)
    print(f"  [CLASSIFY] Метка: '{label}' (confidence={conf:.2f})")
    ann_obj = Annotation(
        document_id=doc_obj.id, label=label,
        confidence=conf, model_name='keyword-baseline-v1')
    session_obj.add(ann_obj)

    session_obj.commit()
    print(f"  [DB] Результаты сохранены в БД")
    return ner_results, (label, conf)

# --- Запуск конвейера ---
ner_out, cls_out = process_document(doc1, session)

# --- Проверка результатов ---
print("\n" + "=" * 60)
print("Проверка: все сущности документа в БД")
print("=" * 60)
all_ents = (session.query(Entity)
            .filter(Entity.document_id == doc1.id)
            .order_by(Entity.ner_type).all())
type_counts = Counter(e.ner_type for e in all_ents)
for ner_type, count in type_counts.items():
    print(f"  {ner_type}: {count}")
print(f"\nВсего сущностей документа ID={doc1.id}: {len(all_ents)}")

## 11. Заключение

В настоящей лекции рассмотрен полный жизненный цикл базы данных юридической информационно-поисковой системы: от формализации требований и концептуального проектирования до физической реализации, обеспечения безопасности, масштабирования и интеграции с аналитическими компонентами.

Центральным результатом лекции является понимание того, что база данных представляет собой не пассивное хранилище, а активный аналитический инструмент, архитектура которого определяется спецификой предметной области и характером типичных запросов. Юридическая информация обладает выраженной дуальной структурой (структурированные метаданные и неструктурированные тексты), что обуславливает необходимость гибридного подхода к организации хранения и поиска.

Реляционная модель данных, реализованная в PostgreSQL, обеспечивает строгую типизацию, ограничения целостности и мощный декларативный язык запросов SQL для работы со структурированными метаданными. Расширения PostgreSQL для полнотекстового поиска ($\texttt{tsvector}$/$\texttt{tsquery}$), хранения JSON-данных и векторных операций ($\texttt{pgvector}$) позволяют реализовать все три парадигмы поиска в рамках единой СУБД.

Обеспечение целостности и безопасности данных является обязательным условием эксплуатации юридической информационной системы. Рассмотренные механизмы включают ограничения целостности, триггеры аудита, ролевую модель доступа, Row-Level Security и резервное копирование с поддержкой PITR. Совокупность этих механизмов составляет минимально необходимый набор защитных мер для промышленной эксплуатации.

Масштабирование базы данных осуществляется посредством комбинации вертикального и горизонтального подходов: репликация обеспечивает масштабирование операций чтения, партиционирование обеспечивает эффективную работу с большими объёмами данных, а материализованные представления позволяют предварительно вычислять аналитические агрегаты.

Интеграция базы данных с NLP-конвейером замыкает архитектурный контур: результаты извлечения сущностей, классификации и векторизации документов сохраняются в базе и используются для интеллектуального поиска, создавая замкнутый цикл «хранение -- обработка -- анализ -- поиск».

---


## Контрольные вопросы для самопроверки

1. В чём заключается принципиальное различие между реляционными и нереляционными моделями данных? Какие критерии определяют выбор модели при проектировании юридической базы данных? Обоснуйте, почему для хранения метаданных судебных дел предпочтительна реляционная модель, а для хранения полных текстов документов может быть оправдано использование документоориентированного хранилища.

2. Объясните назначение и механизм работы полнотекстового поиска в PostgreSQL. В чём преимущество типов $\texttt{tsvector}$/$\texttt{tsquery}$ по сравнению с простым поиском по шаблону ($\texttt{LIKE}$)? Какую роль играет морфологический анализ при поиске в русскоязычных юридических текстах?

3. Какие свойства ACID обеспечивают целостность данных при конкурентном доступе? Приведите пример ситуации в юридической информационной системе, когда нарушение каждого из свойств (атомарности, согласованности, изолированности, долговечности) может привести к практическим проблемам.

4. Опишите стратегию партиционирования таблицы документов по дате. Каков механизм отсечения партиций (partition pruning) и в каких условиях он обеспечивает максимальный выигрыш в производительности? Оцените теоретическое ускорение для базы из $N = 10^7$ документов, распределённых по $P = 10$ партициям, при запросе к одному году данных.

5. Охарактеризуйте принцип наименьших привилегий применительно к управлению доступом в юридической базе данных. Какие роли и привилегии целесообразно определить для аналитика, автоматизированного конвейера обработки и аудитора? Как механизм Row-Level Security может быть использован для ограничения доступа к делам, находящимся в производстве?

---

### Задания для самостоятельной работы

1. Спроектируйте расширенную схему юридической базы данных, добавив сущности «нормативный акт» (с атрибутами: наименование, номер, дата принятия, орган издания) и «ссылка на нормативный акт» (связь документа с конкретной статьёй нормативного акта). Реализуйте DDL-скрипт для PostgreSQL и добавьте необходимые ограничения целостности и индексы.

2. Реализуйте полный цикл работы с базой данных на Python: создание таблиц через SQLAlchemy ORM, наполнение тестовыми данными (не менее 10 дел и 30 документов), выполнение аналитических запросов (распределение документов по типам и годам, поиск наиболее часто упоминаемых организаций), визуализация результатов средствами matplotlib.

3. Напишите триггерную функцию на PL/pgSQL, которая при добавлении нового документа автоматически проверяет наличие связанного дела, обновляет счётчик документов в таблице дел и записывает событие в журнал аудита. Протестируйте триггер на тестовых данных.

---

### Рекомендуемые источники

Codd E.F. *A Relational Model of Data for Large Shared Data Banks* (Communications of the ACM, 1970).

Date C.J. *An Introduction to Database Systems* (8th ed., Pearson, 2004).

PostgreSQL Documentation. *Full Text Search*. URL: https://www.postgresql.org/docs/current/textsearch.html

PostgreSQL Documentation. *Table Partitioning*. URL: https://www.postgresql.org/docs/current/ddl-partitioning.html

SQLAlchemy Documentation. URL: https://docs.sqlalchemy.org

FastAPI Documentation. URL: https://fastapi.tiangolo.com

pgvector -- Open-source vector similarity search for Postgres. URL: https://github.com/pgvector/pgvector

Ramakrishnan R., Gehrke J. *Database Management Systems* (3rd ed., McGraw-Hill, 2003).
